In [1]:
! pip install -q torch-adopt optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.9 MB/s eta 0:00:00


### Hyperparameter Optimization

In [2]:
from typing import Any, Callable, Dict, Literal, Tuple

import optuna
import torch
import torch.optim as optim
import torch.utils.data as data
from adopt import ADOPT
from torch import nn


OptimizerName = Literal["Adam", "AdamW", "ADOPT", "RMSprop", "SGD"]
ModelFactory = Callable[..., nn.Module]
DataLoaderFactory = Callable[..., Tuple[data.DataLoader, data.DataLoader]]
TrainEvalLoop = Callable[..., float]


class ParamOptimizer:
    def __init__(self):
        self.results = {}

    def _get_optimizer_search_space(
        self,
        optimizer_name: OptimizerName,
        trial: optuna.Trial,
    ) -> Dict[str, Any]:
        """Define search spaces for each optimizer."""

        if optimizer_name == 'ADOPT':
            return {
                'lr': trial.suggest_float('lr', 1e-5, 1e-1, log=True),
                'betas': (
                    trial.suggest_float('beta1', 0.8, 0.99),
                    trial.suggest_float('beta2', 0.9, 0.9999)
                ),
                'weight_decay': trial.suggest_float('weight_decay', 0.0, 0.1, log=True),
                'decouple': True,
            }

        elif optimizer_name == 'Adam':
            return {
                'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
                'betas': (
                    trial.suggest_float('beta1', 0.8, 0.99),
                    trial.suggest_float('beta2', 0.9, 0.999)
                ),
                'weight_decay': trial.suggest_float('weight_decay', 0.0, 1e-2, log=True),
            }

        elif optimizer_name == 'AdamW':
            return {
                'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
                'betas': (
                    trial.suggest_float('beta1', 0.8, 0.99),
                    trial.suggest_float('beta2', 0.9, 0.999)
                ),
                'weight_decay': trial.suggest_float('weight_decay', 0.0, 0.1, log=True),
            }

        elif optimizer_name == 'RMSprop':
            return {
                'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
                'alpha': trial.suggest_float('alpha', 0.9, 0.999),
                'weight_decay': trial.suggest_float('weight_decay', 0.0, 1e-2, log=True),
                'momentum': trial.suggest_float('momentum', 0.0, 0.1),
            }

        elif optimizer_name == 'SGD':
            return {
                'lr': trial.suggest_float('lr', 1e-4, 1e-1, log=True),
                'momentum': trial.suggest_float('momentum', 0.0, 0.99),
                'weight_decay': trial.suggest_float('weight_decay', 0.0, 1e-2, log=True),
            }

        else:
            raise ValueError(f"Unknown optimizer: {optimizer_name}")

    def _create_optimizer(
        self,
        optimizer_name: str,
        model: nn.Module,
        **kwargs,
    ) -> optim.Optimizer:
        """Create optimizer instance with given parameters."""

        model_params = model.parameters()

        if optimizer_name == 'SGD':
            return optim.SGD(model_params, **kwargs)
        elif optimizer_name == 'RMSprop':
            return optim.RMSprop(model_params, **kwargs)
        elif optimizer_name == 'Adam':
            return optim.Adam(model_params, **kwargs)
        elif optimizer_name == 'AdamW':
            return optim.AdamW(model_params, **kwargs)
        elif optimizer_name in ['ADOPT', 'ADOPT_L2']:
            return ADOPT(model_params, **kwargs)
        else:
            raise ValueError(f"Unknown optimizer: {optimizer_name}")

    def _objective_factory(
        self,
        optimizer_name: OptimizerName,
        model_factory: ModelFactory,
        dataloader_factory: DataLoaderFactory,
        train_eval_loop: TrainEvalLoop,
        device: str = "cuda",
    ) -> Callable[[optuna.Trial], float]:
        """Create objective function for a specific optimizer."""

        def objective(trial: optuna.Trial) -> float:
            optimizer_params = self._get_optimizer_search_space(optimizer_name, trial)
            model = model_factory().to(device)
            optimizer = self._create_optimizer(optimizer_name, model, **optimizer_params)
            train_loader, val_loader = dataloader_factory()

            val_accuracy = train_eval_loop(
                trial,
                model,
                optimizer,
                train_loader,
                val_loader,
                device
            )

            return val_accuracy

        return objective

    def optimize_optimizer_params(
        self,
        optimizer_name: OptimizerName,
        model_factory: ModelFactory,
        dataloader_factory: DataLoaderFactory,
        train_eval_loop: TrainEvalLoop,
        n_trials: int = 20,
        timeout: int | None = None,
        device: str = "cuda",
    ) -> optuna.Study:
        """Optimize hyperparameters for a specific optimizer on a single task."""

        print(f"\nOptimizing {optimizer_name}...")

        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=42),
            pruner=optuna.pruners.MedianPruner(
                n_startup_trials=5,
                n_warmup_steps=3,
                interval_steps=2,
            )
        )

        objective = self._objective_factory(
            optimizer_name,
            model_factory,
            dataloader_factory,
            train_eval_loop,
            device,
        )
        study.optimize(objective, n_trials=n_trials, timeout=timeout)

        self.results[optimizer_name] = {
            'best_params': study.best_params,
            'best_value': study.best_value,
            'n_trials': len(study.trials),
            'study': study,
        }

        print(f"{optimizer_name} - Best value: {study.best_value:.4f}")
        print(f"{optimizer_name} - Best params: {study.best_params}")

        return study

    def get_best_params_dict(
        self,
        optimizer_name: OptimizerName | None = None
    ) -> Dict[str, Any]:
        """Get the best parameters for an optimizer."""

        if optimizer_name:
            return self.results[optimizer_name]
        else:
            return self.results